## Imports

In [19]:

from hotb_2026.brainaccess_connection import (
    DEVICE_NAME,
    MONTAGE,
    BrainAccessDriver,
    EEGStreamer,
)

## Electrodes Montage
Electrodes Montage is defined as python object. Here you can see default HALO 6 mini montage.

In [20]:
print("Electrodes:")
for electrode in MONTAGE.electrodes:
    print(f"\t{electrode}")

print(f"{MONTAGE.sfreq=}")
print(f"{MONTAGE.bias_electrode=}")

Electrodes:
	Electrode(name='F3', pin=0)
	Electrode(name='F4', pin=1)
	Electrode(name='C3', pin=2)
	Electrode(name='C4', pin=3)
	Electrode(name='P3', pin=4)
	Electrode(name='P4', pin=5)
	Electrode(name='O1', pin=6)
	Electrode(name='O2', pin=7)
MONTAGE.sfreq=250.0
MONTAGE.bias_electrode=7


## Driver
Choose a driver and provide name of device and montage

In [21]:
print(f"Default device name is {DEVICE_NAME}")
driver = BrainAccessDriver(DEVICE_NAME, MONTAGE)


Default device name is BA MINI 078


## Streamer
Object for streaming a signal. Name Stream and set up window time.

In [22]:
streamer = EEGStreamer(
    driver=driver,
    montage=MONTAGE,
    stream_name="TEST Device Stream",
    source_id="eeg-1-test",
    window_s = 5
)


### Preprocessor
`window` has shape `(n_channels, n_samples)` in montage order. `add_processor()` registers a function that gets the current EEG window and its timestamp each time new data arrives.

The `add_preprocessor()` do not modify output signal. It is just added to track what happen with siganl.

In [23]:
from collections import deque

import numpy as np

# Array for getting preprcessed data
cleaned = deque(maxlen=100)  # (window, t) pairs for a consumer thread


def preprocess(window: np.ndarray, t: float) -> None:
    if window.shape[1] < 32:  # not enough samples yet
        return

    x = window.astype(np.float32, copy=True)
    x -= x.mean(axis=1, keepdims=True)   # remove DC per channel

    np.clip(x, -200.0, 200.0, out=x)     # drop obvious artifacts (µV)
    cleaned.append((x, t))


streamer.add_processor(preprocess)

## Getting Signal
`is_full` is a flag.

In [24]:
streamer.start()
try:
    while streamer.running:
        if streamer.is_full:
            print(streamer.window())
            print()
finally:
    streamer.stop()


[2026-11-14 15:43:40.308] [tid  56126] =============[4.0.0]==============
[2026-11-14 15:43:40.308] [tid  56126] Adapter index: 0
[2026-11-14 15:43:40.308] [tid  56126] Chunk size: 25 Logs enabled ?: 1
[2026-11-14 15:43:40.308] [tid  56126] Update path: ../updates/firmware_HALO.bin 
[2026-11-14 15:43:40.308] [tid  56126] Log : path: bacore.log
[2026-11-14 15:43:40.308] [tid  56126]   buffer size: 512 autoflush ?: 0
[2026-11-14 15:43:40.308] [tid  56126]   append ?: 0 level: 2
[2026-11-14 15:43:40.308] [tid  56126]   timestamps enabled ?: 1 thread IDs enabled ?: 1
[2026-11-14 15:43:40.308] [tid  56126] ==================================
[2026-11-14 15:43:40.309] [tid  56126] Connecting to : BA MINI 078
[2026-11-14 15:43:40.309] [tid  56126] Connection attempt 0
[2026-11-14 15:43:46.352] [tid  56126] Bluetooth connection established! Fetching device info...
[2026-11-14 15:43:46.569] [tid  56126] Model: 0 Electrode count: 8
[2026-11-14 15:43:46.589] [tid  56126] Setting stream rate 6
[202

KeyboardInterrupt: 